### Convert to coNLL

In [ ]:
# Actual train

import pandas as pd
from pathlib import Path
import spacy
import os
import gc

def note_to_conll(note_path, spans, label, nlp):
    """
    Read one note, tokenize it, and tag only EVENT entities as EVENT, everything else as O.
    spans: list of (start, end, entity_type)
    """
    text = Path(note_path).read_text(encoding="utf-8")
    doc = nlp(text)

    tok_data = [(tok.text, tok.idx, tok.idx + len(tok.text)) for tok in doc]
    labs = ["O"] * len(tok_data)

    for start, end, typ in spans:
        if typ != "EVENT":
            continue
        for i, (_, t_start, t_end) in enumerate(tok_data):
            if t_end <= start or t_start >= end:
                continue
            labs[i] = label

    return [f"{tok}\t{lab}" for (tok, _, _), lab in zip(tok_data, labs)]

def conll_helper(dfx, sub_dir, label):
    
    for note_path, group in dfx.groupby("note_path"):
        spans = list(zip(group.span_start, group.span_end, group.entity_type))
        tokens = note_to_conll(note_path, spans, label, nlp)
        
        if not any(line.endswith(f"\t{label}") for line in tokens):
            continue

        fname = Path(note_path).name
        subpath = Path(*Path(note_path).parts[-5:-1])
        if sub_dir == 'event':
            out_dir = Path(f"../conll_data/test/{sub_dir}") / subpath
        else:
            out_dir = Path(f"../conll_data/test/modifiers/{sub_dir}/{label}") / subpath
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = out_dir / Path(fname).with_suffix(".conll")
        out_file.write_text("\n".join(tokens), encoding="utf-8")


df = pd.read_csv("../entity_tags_test.csv")
df = df.dropna(subset=['entity_type'])
# df = df[df['entity_type']=='EVENT'].dropna(subset=["polarity", "contextualModality"])
# pol_vals = df['polarity'].unique()
# mod_vals = df['contextualModality'].unique()

nlp = spacy.load("en_core_sci_md")

conll_helper(df[df['entity_type']=='EVENT'], 'event', 'EVENT')

# for p in pol_vals:
#     p_df = df[df['polarity']==p]
#     conll_helper(p_df, "polarity", p)
    
# for m in mod_vals:
#     m_df = df[df['contextualModality']==m]
#     conll_helper(m_df, "modality", m)
    
    
print("Success")

del nlp
gc.collect()

## EVENTs NER

### Finetuning clinicalBERT

In [ ]:
import os
import re
from glob import glob
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from sklearn.metrics import (
     classification_report,
     precision_recall_fscore_support,
     accuracy_score,
     confusion_matrix,
)
import torch
torch.cuda.empty_cache()
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL_DIR = "../../../nlp_models/bert_models/clinicalBERT_local/"
SAVE_DIR = "../finedtuned_bert_events_experiment2/"
BATCH_SIZE = 64

# 1. Read a CoNLL file into [{"tokens":…, "labels":…}, …]
def read_conll_file(path):
    examples, tokens, labels = [], [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    examples.append({"tokens": tokens, "labels": labels})
                    tokens, labels = [], []
                    
            if "\t" in line:
                parts = line.split("\t")
            else:
                parts = re.split(r"\s+", line, maxsplit=1)
            if len(parts) < 2:
                continue

            token, label = parts[0], parts[-1]
            tokens.append(token)
            labels.append(label)
            
        if tokens:
            examples.append({"tokens": tokens, "labels": labels})
            
    return examples

# 2. Get file paths for each experiment
def get_paths(experiment):
    actual_train = glob("../conll_data/events/*/train/**/*.conll", recursive=True)
    dev_paths = glob("../conll_data/events/*/dev/**/*.conll", recursive=True)
    synthetic_train = glob("../conll_data/synthetic2/events/*/*.conll")
#     synthetic_train += glob("../conll_data/synthetic2/*/*.conll")
    
    if experiment == "actual":
        train_paths = actual_train
    elif experiment == "synthetic":
        train_paths = synthetic_train
    elif experiment == "both":
        train_paths = actual_train + synthetic_train
    else:
        raise ValueError(f"Unknown experiment {experiment}")
        
    return train_paths, dev_paths

# 3. Main loop over experiments
experiments = ["actual", "synthetic", "both"]
for exp in experiments:
    print(f"\n\n===== Experiment: {exp} =====")
    train_files, dev_files = get_paths(exp)

    # 3a. Read examples
    train_exs = [ex for p in train_files for ex in read_conll_file(p)]
    dev_exs = [ex for p in dev_files for ex in read_conll_file(p)]

    # 3b. Split train → train / valid (90/10)
    train_exs, valid_exs = train_test_split(train_exs, test_size=0.1, random_state=42)

    # 3c. Build label list (["EVENT","O"])
    all_labels = sorted({lab for ex in train_exs + valid_exs + dev_exs for lab in ex["labels"]})
    label2id   = {l:i for i,l in enumerate(all_labels)}
    id2label   = {i:l for l,i in label2id.items()}

    for ex in (train_exs + valid_exs + dev_exs):
        ex["label_ids"] = [label2id[l] for l in ex["labels"]]

    # 3e. Create Hugging Face datasets
    ds_train = Dataset.from_list([{"tokens":ex["tokens"], "labels":ex["label_ids"]} for ex in train_exs])
    ds_valid = Dataset.from_list([{"tokens":ex["tokens"], "labels":ex["label_ids"]} for ex in valid_exs])
    ds_test  = Dataset.from_list([{"tokens":ex["tokens"], "labels":ex["label_ids"]} for ex in dev_exs])
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
    tokenizer.model_max_length = 512
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_DIR,
        num_labels=len(all_labels),
        id2label=id2label,
        label2id=label2id,
    )
    data_collator = DataCollatorForTokenClassification(tokenizer)

    # 4. Tokenize & align labels
    def tokenize_and_align(ex):
        toks = tokenizer(ex["tokens"], is_split_into_words=True, truncation=True)
        word_ids = toks.word_ids()
        prev_word_idx = None
        lab_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                lab_ids.append(-100)
            elif word_idx != prev_word_idx:
                lab_ids.append(ex["labels"][word_idx])
            else:
                lab_ids.append(ex["labels"][word_idx])
            prev_word_idx = word_idx
        toks["labels"] = lab_ids
        return toks

    ds_train = ds_train.map(tokenize_and_align, batched=False)
    ds_valid = ds_valid.map(tokenize_and_align, batched=False)
    ds_test  = ds_test.map(tokenize_and_align, batched=False)

    # 5. Metrics callback using seqeval
    def compute_metrics(p):
        logits, label_ids = p
        preds = np.argmax(logits, axis=2).flatten()
        labels = label_ids.flatten()

        # mask out subword positions
        mask = labels != -100
        true_labels = labels[mask]
        true_preds  = preds[mask]

        precision, recall, f1, _ = precision_recall_fscore_support(
            true_labels, true_preds,
            labels=[label2id["EVENT"] ],
            average="binary",
            zero_division=0,
        )
        acc = accuracy_score(true_labels, true_preds)

        return {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "accuracy": acc
        }
    
    # 6. Training arguments
    args = TrainingArguments(
        output_dir=f"{SAVE_DIR}/{exp}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to=[],
    )

    # 7. Train
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_train,
        eval_dataset=ds_valid,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.train()
    tokenizer.save_pretrained(f"{SAVE_DIR}/{exp}")
    trainer.save_model(f"{SAVE_DIR}/{exp}")

    print(f"\nEvaluating on dev for {exp}")
    preds_output = trainer.predict(ds_test)
    logits, label_ids = preds_output.predictions, preds_output.label_ids
    preds = np.argmax(logits, axis=2)

    mask = label_ids != -100
    flat_true_ids = label_ids[mask]
    flat_pred_ids = preds[mask]

    event_id = label2id["EVENT"]
    flat_true = [int(id_ == event_id) for id_ in flat_true_ids]
    flat_pred = [int(id_ == event_id) for id_ in flat_pred_ids]
    
    names = ["O", "EVENT"]
    report = classification_report(
        flat_true,
        flat_pred,
        labels=[0,1],
        target_names=names,
        digits=4
    )
    print(f"\nClassification Report for {exp}:\n{report}")
    num_labels = len(all_labels)
    cm = confusion_matrix(
        flat_true_ids,
        flat_pred_ids,
        labels=list(range(num_labels))
    )

    fig, ax = plt.subplots()
    cax = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    fig.colorbar(cax)

    n_rows, n_cols = cm.shape
    for i in range(n_rows):
        for j in range(n_cols):
            ax.text(
                j, i,           
                f"{cm[i, j]:,}",
                ha="center",
                va="center",
                color="black"
            )

    ax.set_xticks(np.arange(n_cols))
    ax.set_yticks(np.arange(n_rows))
    ax.set_xticklabels(["EVENT", "O"])
    ax.set_yticklabels(["EVENT", "O"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{exp} Confusion Matrix")
    plt.tight_layout()
    plt.show()

### Inference

In [ ]:
# Inference EVENT

import subprocess
import os
import re
import pandas as pd
import glob
from tqdm import tqdm
import spacy
from medspacy.sentence_splitting import PyRuSHSentencizer
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline


MODEL_DIR = "../finedtuned_bert_events_experiment2/both"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR, device_map="auto")
nlp_ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple", framework="pt")
nlp_sent = spacy.blank("en")
nlp_sent.add_pipe("medspacy_pyrush")


def ner_events(text):
    """
    Perform event-level named entity recognition on the input text.

    Splits the input into sentences using MedSpaCy’s PyRuSH sentence splitter,
    then applies a HuggingFace NER pipeline to each sentence. Returns a list of
    event entities with their character offsets in the original text.

    Args:
        text (str): The raw document text to analyze for event entities.

    Returns:
        List[dict]: A list of dictionaries, each containing:
            - "entity" (str): The text of the detected entity.
            - "start" (int): The start character index in the original text.
            - "end" (int): The end character index in the original text.
            - "type" (str): The entity label/group assigned by the NER model.
    """
    ents = []
    doc = nlp_sent(text)
    for sent in doc.sents:
        sent_text = sent.text
        offset = sent.start_char
        for e in nlp_ner(sent_text):
            ents.append({
                "entity": e["word"],
                "start": offset + e["start"],
                "end": offset + e["end"],
                "type": e.get("entity_group")
            })
    return ents

# df = pd.read_csv('../entity_tags_test.csv')
entity_list = []
# dev_pths = df[df['split_type']=='test']['note_path'].unique()
# dev_pths = glob.glob("/home/jainv/dr_osb_lab/ChemoTask/chemoTimelines2025_test_data/subtask2/Patient_Notes/**/*.txt", recursive=True)
dev_pths = glob.glob("/home/jainv/dr_osb_lab/ChemoTask/chemoTimelines2024_train_dev_labeled/subtask2/Patient_Notes/**/*.txt", recursive=True)
print("Total notes: ", len(dev_pths))

for pth in tqdm(dev_pths):
    text = open(pth, 'r').read()
    events_list = ner_events(text)
    if len(events_list) == 0:
        print("No events detected.")
    entity_list.append(
        {
            "note_path" : pth
            "events_list" : events_list,
        }
    )

pd.DataFrame.from_dict(entity_list).to_csv("../dev_entities_events_subtask2.csv", index=False)
print("NER complete")

## TIMEX3

In [ ]:
import multiprocessing as mp
import subprocess
import re
import pandas as pd

def extract_anchor_date(text):
    match = re.search(r'Principal Date\.{2,}(\d{8})', text)
    if match:
        date_str = match.group(1)
        year = int(date_str[:4])
        month = int(date_str[4:6])
        day = int(date_str[6:8])
        return year, month, day
    else:
        raise ValueError("Anchor date not found")


def call_timenorm_cli(input_pth, jar_path, year, month, day):
    """
    Invoke the TimeNorm Java CLI to detect and normalize temporal expressions.

    Builds and runs a subprocess command: `java -jar <jar_path> <input_path> <year> <month> <day>`.
    Captures and returns the CLI’s stdout. Raises if the process fails.

    Args:
        input_path (str): Path to the input text file for TimeNorm.
        jar_path (str): Path to the TimeNorm fat JAR.
        year (int): Anchor year.
        month (int): Anchor month.
        day (int): Anchor day.

    Returns:
        str: The raw stdout output from the TimeNorm CLI.

    Raises:
        RuntimeError: If the TimeNorm process exits with a non-zero code.
    """
    cmd = [
        'java', '-jar', jar_path,
        input_pth,
        str(year), str(month), str(day)
    ]
    p = subprocess.run(cmd, capture_output=True, text=True)

    if p.returncode != 0:
        raise RuntimeError(f"TimeNorm error: {p.stderr}")

    return p.stdout


def ner_timex3(text, pth, jar_pth):
    """
    Detect and normalize TIMEX3 (temporal) expressions in the input text.

    1. Extracts an anchor date from the text.
    2. Calls the TimeNorm CLI to identify and normalize temporal spans.
    3. Parses the CLI output to build a list of unique TIMEX3 entries.

    Each entry is a dict with keys:
      - "entity": the original span text
      - "start": character start index
      - "end": character end index
      - "normalized": list of all normalized time intervals

    Args:
        text (str): The raw document text containing temporal expressions.

    Returns:
        List[dict]: A list of TIMEX3 dictionaries as described above.

    Raises:
        Exception: Wraps and re-raises errors encountered during extraction or CLI execution.
    """
    try:
        year, month, day = extract_anchor_date(text)
        result = call_timenorm_cli(pth, jar_pth, year, month, day)

        pattern = re.compile(
            r"NL span=\((?P<start>\d+),(?P<end>\d+)\) text='(?P<text>.*?)' → \[(?P<norm>[^\]]*)\]"
        )
        entries = {}
        for line in result.splitlines():
            if line.startswith("NL span") or line.startswith("ISO span"):
                m = pattern.match(line.strip())
                if not m:
                    continue
                start = int(m.group('start'))
                end = int(m.group('end'))
                text = m.group('text')
                norm = m.group('norm')
                key = (start, end, text)

                if key not in entries and norm != 'cannot normalize':
                    entries[key] = {
                        'entity': text,
                        'start': start,
                        'end': end,
                        'normalized': []
                    }
                else:
                    continue

                if norm not in entries[key]['normalized']:
                    entries[key]['normalized'].append(norm)

        return {
            "note_path" : pth
            "timex_list" : list(entries.values())
        }

    except Exception as e:
        print(e)
        return None

jar = "/data/user/home/jainv/ChemoTask/chemotimelines_timenorm/timenorm_remy/target/scala-2.13/timenorm_remy-assembly-0.1.0.jar"
df = pd.read_csv("/data/user/home/jainv/ChemoTask/entity_tags.csv")
input_paths = set(df[df['split_type']=='dev']['note_path'].to_list())
files_and_jars = [(pth, jar) for pth in input_paths]

with mp.Pool(processes=8) as pool:
    results = pool.map(ner_timex3, files_and_jars)

res = [res for res in results if res is not None]
pd.DataFrame.from_records(res).to_csv('/data/user/home/jainv/ChemoTask/dev_entities_timex.csv')